In [1]:
import json
from math import sqrt
from pathlib import Path

import pandas as pd
from scipy import stats

from bcbench.results.base import create_result_from_json

result_folder = Path("result")


def load_results_df(setup_folder: Path) -> pd.DataFrame:
    rows = []
    for jsonl_file in setup_folder.glob("*.jsonl"):
        run_id = jsonl_file.stem
        for line in jsonl_file.read_text(encoding="utf-8").splitlines():
            r = create_result_from_json(json.loads(line))
            data = r.model_dump()
            metrics = data.pop("metrics", {}) or {}
            tool_usage = metrics.pop("tool_usage", {}) or {}
            rows.append({"run_id": run_id, **data, **metrics, **tool_usage})
    return pd.DataFrame(rows)


def compute_summary_stats(df: pd.DataFrame) -> dict:
    n_runs = df["run_id"].nunique()
    per_run_resolved = df.groupby("run_id")["resolved"].mean() * 100
    mean_resolved = per_run_resolved.mean()

    # 95% CI using t-distribution
    sem = per_run_resolved.std(ddof=1) / sqrt(n_runs)
    t_95 = stats.t.ppf(0.975, df=n_runs - 1)
    ci_half = t_95 * sem

    return {
        "n_runs": n_runs,
        "n_results": len(df),
        "n_instances": df["instance_id"].nunique(),
        "avg_execution_time": df["execution_time"].mean(),
        "mean_resolved": mean_resolved,
        "ci_95_half": ci_half,
        "ci_95_low": mean_resolved - ci_half,
        "ci_95_high": mean_resolved + ci_half,
    }

In [2]:
# Discover and load all setup results
all_results: dict[str, pd.DataFrame] = {}

for setup_folder in sorted(result_folder.iterdir()):
    if setup_folder.is_dir():
        df = load_results_df(setup_folder)
        if not df.empty:
            all_results[setup_folder.name] = df

summary_data = []
for _setup_name, df in all_results.items():
    model_stats = compute_summary_stats(df)
    agent = df["agent_name"].iloc[0] if "agent_name" in df.columns else "Unknown"
    summary_data.append(
        {
            "Model": df["model"].iloc[0],
            "Agent": agent,
            "Runs": model_stats["n_runs"],
            "Instances": model_stats["n_instances"],
            "Mean Resolved %": f"{model_stats['mean_resolved']:.2f}",
            "95% CI ±": f"{model_stats['ci_95_half']:.2f}",
            "Avg Time (s)": f"{model_stats['avg_execution_time']:.0f}",
        }
    )

summary_df = pd.DataFrame(summary_data)
summary_df

,Model,Agent,Runs,Instances,Mean Resolved %,95% CI ±,Avg Time (s)
0,gpt-5-1-codex-max,GitHub Copilot CLI,10,55,50.91,3.13,152
1,claude-haiku-4-5,GitHub Copilot CLI,11,55,45.12,2.49,150
2,claude-haiku-4-5,mini-bc-agent,10,55,36.55,4.44,193
3,claude-opus-4-5,mini-bc-agent,10,55,62.18,2.11,331
4,claude-opus-4-5,GitHub Copilot CLI,10,55,59.64,1.60,134


In [3]:
# Select a specific model for detailed analysis (using haiku-4-5 as example)
ANALYSIS_MODEL = "haiku-4-5"

haiku_df = all_results[ANALYSIS_MODEL]
haiku_stats = compute_summary_stats(haiku_df)

print(f"Detailed Analysis: {ANALYSIS_MODEL} (GitHub Copilot CLI)")
print(f"{'=' * 60}")
print(f"Runs: {haiku_stats['n_runs']} | Instances: {haiku_stats['n_instances']}")
print(f"Average execution time: {haiku_stats['avg_execution_time']:.1f}s ({haiku_stats['avg_execution_time'] / 60:.1f}m)")
print(f"Mean resolved: {haiku_stats['mean_resolved']:.2f}% (95% CI: ±{haiku_stats['ci_95_half']:.2f}%)")
print(f"Category: {haiku_df['category'].iloc[0]}")
print(f"Agent: {haiku_df['agent_name'].iloc[0]}")

# Prepare DataFrame for detailed analysis (drop common metadata columns)
haiku_df = haiku_df.drop(columns=["category", "timeout", "error_message", "agent_name", "model", "experiment"])
haiku_df.head(n=1)

Detailed Analysis: haiku-4-5 (GitHub Copilot CLI)
Runs: 11 | Instances: 55
Average execution time: 149.7s (2.5m)
Mean resolved: 45.12% (95% CI: ±2.49%)
Category: EvaluationCategory.BUG_FIX
Agent: GitHub Copilot CLI


,run_id,instance_id,project,resolved,build,generated_patch,execution_time,llm_duration,turn_count,prompt_tokens,...,report_intent,view,glob,grep,edit,powershell,create,read_powershell,stop_powershell,bash
0,20104171950,microsoft__BCApps-4699,Shopify,True,True,diff --git a/src/Apps/W1/Shopify/App/src/Produ...,110.779,103.416,37,1000000,...,5,17,4.0,11.0,3.0,1.0,NaN,NaN,NaN,NaN


In [4]:
import plotly.graph_objects as go

run_ids: list[str] = sorted(haiku_df["run_id"].unique())

# Calculate observed SD from all runs (this is our estimate of true population SD)
all_per_run = haiku_df.groupby("run_id")["resolved"].mean() * 100
observed_sd: float = all_per_run.std(ddof=1)

cumulative_data = []
for n in range(4, len(run_ids) + 1):  # Start from 4 runs
    subset = haiku_df[haiku_df["run_id"].isin(run_ids[:n])]
    per_run = subset.groupby("run_id")["resolved"].mean() * 100

    sem = per_run.std(ddof=1) / sqrt(n)

    # Use t-distribution for small samples (more accurate than z=1.96)
    t_95 = stats.t.ppf(0.975, df=n - 1)
    t_90 = stats.t.ppf(0.95, df=n - 1)

    ci_half_95 = t_95 * sem
    ci_half_90 = t_90 * sem

    # Minimum Detectable Effect (MDE) at 80% power for two-sample comparison
    # For comparing two setups with equal variance and equal n:
    # MDE = (t_alpha + t_beta) * SD * sqrt(2/n)
    # where t_beta = 0.84 for 80% power (one-sided)
    t_beta = stats.norm.ppf(0.80)  # ~0.84 for 80% power
    mde_95 = (t_95 + t_beta) * observed_sd * sqrt(2 / n)
    mde_90 = (t_90 + t_beta) * observed_sd * sqrt(2 / n)

    cumulative_data.append(
        {
            "n_runs": n,
            "sem": sem,
            "ci_half_95": ci_half_95,
            "ci_half_90": ci_half_90,
            "mde_95": mde_95,
            "mde_90": mde_90,
        }
    )

analysis_df = pd.DataFrame(cumulative_data)

fig = go.Figure()

# 95% CI half-width
fig.add_trace(
    go.Scatter(
        x=analysis_df["n_runs"],
        y=analysis_df["ci_half_95"],
        mode="lines+markers",
        name="95% CI Half-Width (±%)",
        line={"color": "blue", "width": 2},
        marker={"size": 8},
    )
)

# 90% CI half-width
fig.add_trace(
    go.Scatter(
        x=analysis_df["n_runs"],
        y=analysis_df["ci_half_90"],
        mode="lines+markers",
        name="90% CI Half-Width (±%)",
        line={"color": "lightblue", "width": 2, "dash": "dash"},
        marker={"size": 8},
    )
)

# MDE at 80% power (95% significance)
fig.add_trace(
    go.Scatter(
        x=analysis_df["n_runs"],
        y=analysis_df["mde_95"],
        mode="lines+markers",
        name="MDE @ 80% Power (alpha=0.05)",
        line={"color": "red", "width": 2},
        marker={"size": 8, "symbol": "diamond"},
    )
)

# MDE at 80% power (90% significance)
fig.add_trace(
    go.Scatter(
        x=analysis_df["n_runs"],
        y=analysis_df["mde_90"],
        mode="lines+markers",
        name="MDE @ 80% Power (alpha=0.10)",
        line={"color": "salmon", "width": 2, "dash": "dash"},
        marker={"size": 8, "symbol": "diamond"},
    )
)

# Reference thresholds
for threshold, color in [(5, "green"), (10, "orange")]:
    fig.add_hline(
        y=threshold,
        line_dash="dot",
        line_color=color,
        annotation_text=f"{threshold}% difference",
        annotation_position="bottom right",
    )

fig.update_layout(
    title=f"Run Count Analysis for Confident Comparisons<br><sup>Observed SD across runs: {observed_sd:.2f}%</sup>",
    xaxis_title="Number of Runs per Setup",
    yaxis_title="Effect Size (%)",
    xaxis={"tickmode": "linear", "tick0": 4, "dtick": 1},
    yaxis={"rangemode": "tozero"},
    template="plotly_white",
    legend={"yanchor": "top", "y": 0.99, "xanchor": "right", "x": 0.99},
    height=500,
)
fig.show()

print("\nHow many runs do you need?")
print("=" * 70)
print(f"{'Runs':<6} {'95% CI ±':<12} {'90% CI ±':<12} {'MDE (a=.05)':<14} {'MDE (a=.10)':<14}")
print("-" * 70)
for _, row in analysis_df.iterrows():
    print(f"{int(row['n_runs']):<6} {row['ci_half_95']:.2f}%{'':<6} {row['ci_half_90']:.2f}%{'':<6} {row['mde_95']:.2f}%{'':<8} {row['mde_90']:.2f}%")
print("-" * 70)
print("\nInterpretation:")
print("  • CI Half-Width: Your mean estimate will be ± this value")
print("  • MDE (Minimum Detectable Effect): Smallest difference you can detect")
print("    between two setups with 80% power at the given significance level")
print(f"\nTo detect a 5% difference with 80% power (a=0.05): ~{analysis_df[analysis_df['mde_95'] <= 5]['n_runs'].min() if any(analysis_df['mde_95'] <= 5) else '>11'} runs per setup")


How many runs do you need?
Runs   95% CI ±     90% CI ±     MDE (a=.05)    MDE (a=.10)   
----------------------------------------------------------------------
4      8.96%       6.62%       10.56%         8.38%
5      6.10%       4.68%       8.49%         6.98%
6      4.63%       3.63%       7.31%         6.12%
7      4.05%       3.22%       6.52%         5.52%
8      3.45%       2.76%       5.95%         5.08%
9      3.00%       2.42%       5.50%         4.72%
10     2.67%       2.17%       5.15%         4.44%
11     2.49%       2.03%       4.86%         4.20%
----------------------------------------------------------------------

Interpretation:
  • CI Half-Width: Your mean estimate will be ± this value
  • MDE (Minimum Detectable Effect): Smallest difference you can detect
    between two setups with 80% power at the given significance level

To detect a 5% difference with 80% power (a=0.05): ~11 runs per setup


In [5]:
from bcbench.config import get_config
from bcbench.dataset import DatasetEntry, load_dataset_entries

_config = get_config()
bcbench_dataset: list[DatasetEntry] = load_dataset_entries(_config.paths.dataset_path)

dataset_df = pd.DataFrame(
    [
        {
            "instance_id": entry.instance_id,
            "image_count": entry.metadata.image_count or 0,
            "area": entry.metadata.area or "Unknown",
            "gold_patch": entry.patch,
        }
        for entry in bcbench_dataset
    ]
)

haiku_merged_df = haiku_df.merge(dataset_df, on="instance_id", how="left")
print(f"Merged {len(haiku_merged_df)} results with dataset metadata")
haiku_merged_df.head(n=1)

Merged 605 results with dataset metadata


,run_id,instance_id,project,resolved,build,generated_patch,execution_time,llm_duration,turn_count,prompt_tokens,...,grep,edit,powershell,create,read_powershell,stop_powershell,bash,image_count,area,gold_patch
0,20104171950,microsoft__BCApps-4699,Shopify,True,True,diff --git a/src/Apps/W1/Shopify/App/src/Produ...,110.779,103.416,37,1000000,...,11.0,3.0,1.0,NaN,NaN,NaN,NaN,5,shopify,diff --git a/src/Apps/W1/Shopify/App/src/Produ...


In [6]:
from unidiff import PatchSet


def count_files_in_patch(patch: str) -> int:
    return len(PatchSet(patch))


haiku_merged_df["expected_files"] = haiku_merged_df["gold_patch"].apply(count_files_in_patch)

bins = [0, 1, 2, float("inf")]
labels = ["1", "2", "3+"]
haiku_merged_df["files_bin"] = pd.cut(haiku_merged_df["expected_files"], bins=bins, labels=labels)

instance_files_df = (
    haiku_merged_df.groupby("instance_id")
    .agg(
        files_bin=("files_bin", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

files_resolution_df = (
    instance_files_df.groupby("files_bin", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

total_instances = files_resolution_df["count"].sum()
files_resolution_df["% Resolved"] = (files_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
files_resolution_df["Instances"] = files_resolution_df["count"].astype(str) + " (" + (files_resolution_df["count"] / total_instances * 100).round(1).astype(str) + "%)"
files_resolution_df = files_resolution_df.rename(columns={"files_bin": "Files Changed"})

print('Average "% Resolved" by Number of Files Changed (gold patch):')
print(files_resolution_df[["Files Changed", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Number of Files Changed (gold patch):
Files Changed % Resolved  Instances
            1      49.9% 47 (85.5%)
            2       7.3%   5 (9.1%)
           3+      33.3%   3 (5.5%)


In [7]:
def count_loc_in_patch(patch: str) -> int:
    patch_set = PatchSet(patch)
    return patch_set.added + patch_set.removed


haiku_merged_df["expected_loc"] = haiku_merged_df["gold_patch"].apply(count_loc_in_patch)

bins = [0, 5, 10, 50, float("inf")]
labels = ["1-5", "6-10", "11-50", "50+"]
haiku_merged_df["loc_bin"] = pd.cut(haiku_merged_df["expected_loc"], bins=bins, labels=labels)

instance_loc_df = (
    haiku_merged_df.groupby("instance_id")
    .agg(
        loc_bin=("loc_bin", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

loc_resolution_df = (
    instance_loc_df.groupby("loc_bin", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

loc_resolution_df["% Resolved"] = (loc_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
loc_resolution_df["Instances"] = loc_resolution_df["count"].astype(str) + " (" + (loc_resolution_df["count"] / total_instances * 100).round(1).astype(str) + "%)"
loc_resolution_df = loc_resolution_df.rename(columns={"loc_bin": "LoC Changed"})

print('Average "% Resolved" by Lines of Code Changed (gold patch):')
print(loc_resolution_df[["LoC Changed", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Lines of Code Changed (gold patch):
LoC Changed % Resolved  Instances
        1-5      64.5% 22 (40.0%)
       6-10      42.4%  6 (10.9%)
      11-50      33.8% 21 (38.2%)
        50+      16.7%  6 (10.9%)


In [8]:
haiku_merged_df["project_group"] = haiku_merged_df["project"].apply(lambda x: "BaseApp" if x == "BaseApp" else "First-party Apps")

instance_project_df = (
    haiku_merged_df.groupby("instance_id")
    .agg(
        project_group=("project_group", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

project_resolution_df = (
    instance_project_df.groupby("project_group", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

project_resolution_df["% Resolved"] = (project_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
project_resolution_df["Instances"] = project_resolution_df["count"].astype(str) + " (" + (project_resolution_df["count"] / total_instances * 100).round(1).astype(str) + "%)"
project_resolution_df = project_resolution_df.rename(columns={"project_group": "Project Group"})

print('Average "% Resolved" by Project Group:')
print(project_resolution_df[["Project Group", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Project Group:
   Project Group % Resolved  Instances
         BaseApp      44.2% 45 (81.8%)
First-party Apps      49.1% 10 (18.2%)


In [9]:
bins = [-1, 0, 5, 10, float("inf")]
labels = ["0", "1-5", "6-10", "10+"]
haiku_merged_df["image_bin"] = pd.cut(haiku_merged_df["image_count"], bins=bins, labels=labels)

instance_image_df = (
    haiku_merged_df.groupby("instance_id")
    .agg(
        image_bin=("image_bin", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

image_resolution_df = (
    instance_image_df.groupby("image_bin", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

image_resolution_df["% Resolved"] = (image_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
image_resolution_df["Instances"] = image_resolution_df["count"].astype(str) + " (" + (image_resolution_df["count"] / total_instances * 100).round(1).astype(str) + "%)"
image_resolution_df = image_resolution_df.rename(columns={"image_bin": "Image Count"})

print('Average "% Resolved" by Image Count:')
print(image_resolution_df[["Image Count", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Image Count:
Image Count % Resolved  Instances
          0      48.3% 26 (47.3%)
        1-5      42.4%  9 (16.4%)
       6-10      39.9% 13 (23.6%)
        10+      46.8%  7 (12.7%)
